# Ecoprovinces_ExtendData (Feb 24, 2022)

Code by Caroline Juang (c.juang@columbia.edu)

Use this code to create annual timeseries of forested burned area, for Bailey's ecoprovinces and for EPA ecoregions. The code exports data to the `data` folder.

This is data preparation for the `Corr_BurnedAreaSSTs` notebook.

Instructions: 
* to get the area burned within a certain time period, change the months encompassed in the `is_fireseason` script.
* output is a sum of burned area by the months from `is_fireseason`, for ecoprovinces and ecoregions, exported as TXT files.

## Import data

In [1]:
# import
import numpy as np
import xarray as xr
import pandas as pd
import rioxarray as rio
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 200
from datetime import datetime

In [2]:
# importing data string
directory = 'your_data_directory' # customize this

## scripts

In [3]:
# this is cool, this is for fire season

def is_fireseason(month):
    """
    For specifically selecting the fire season.
    The usage should find the months like: data.sel(time=is_fireseason(data['time.month']))
    """
    return (month >= 1) & (month <= 12)

In [4]:
# add up burned area


def annual_seasonArea(data, thisregion):
    """
    This intakes the ecoregion/ecoprovince (thisregion) and the variable (data), and outputs an
    xarray DataArray with the yearly sums of burned area.
    """
    data = data*thisregion # get in the region
    # only fire season, get annual burn
    timeseries = data.sel(time=is_fireseason(data['time.month'])).resample(indexer={'time':'YE'}).sum(skipna=True)
    timeseries = timeseries.sum(dim=['Y','X']).values
    return timeseries

# Get MODIS and WUMI timeseries by Bailey's Ecoprovinces (WUMI1+WUMI2)

In [ ]:
# import burned area MODIS
#modis_string = directory + '1km\\wildfire\\burnarea_MODISv6.nc'
modis_string = directory + '1km//wildfire//burnarea_MODISv6.nc'
modis_data = xr.open_dataset(modis_string, engine='netcdf4')
modis = modis_data.burnarea
modis_data = [] # clear import

# import forest area
# forest_string = directory + '1km\\landcover\\US_ForestType_Ruefenacht\\forest_type_frac.nc'
forest_string = directory + '1km//landcover//US_ForestType_Ruefenacht//forest_type_frac.nc'
forest = xr.open_dataset(forest_string, engine='netcdf4')
forest = forest.forest_type_frac.sum(dim='ftype') # sum over all forest types

# calculate burned area in forests
modisforest = modis*forest

# calculate non-forest
modisnon = modis - modisforest

# clear forest
del forest

In [ ]:
# Bailey's ecoprovinces (Littell et al., 2009)
# ecoprov_string = directory + '1km\\landcover\ecoregions\\bailey_ecoprovince.nc'
ecoprov_string = directory + '1km//landcover//ecoregions//bailey_ecoprovince.nc'
ecoprov = xr.open_dataset(ecoprov_string, engine='netcdf4')
ecoprov = ecoprov.bailey_ecoprovince

## create timeseries of burned area for each ecoprovince (long runtime)

In [ ]:
# find sum of burned area MODIS for each ecoprovince
firstyear = pd.DatetimeIndex(modis.time.values).year.min()
finalyear = pd.DatetimeIndex(modis.time.values).year.max()
years2 = np.arange(firstyear, finalyear+1, 1)

# initialize list
annualburnall = np.zeros([len(years2), len(ecoprov.province)])
annualburnfor = np.zeros([len(years2), len(ecoprov.province)]) # forest area only
annualburnnon = np.zeros([len(years2), len(ecoprov.province)]) # forest area only

# this takes a while because it's at 1km resolution
counter = 0
for thisregion in ecoprov:
    annualburnall[:,counter] = annual_seasonArea(modis, thisregion)
    annualburnfor[:,counter] = annual_seasonArea(modisforest, thisregion)
    annualburnnon[:,counter] = annual_seasonArea(modisnon, thisregion)
    
    print('province '+str(counter)+'/'+str((len(ecoprov.province)-1))) # progress
    counter+=1
    
# create pandas dataframe
annualburnall_modis_df = pd.DataFrame(annualburnall, index=years2, columns=ecoprov.province.values)
annualburnfor_modis_df = pd.DataFrame(annualburnfor, index=years2, columns=ecoprov.province.values)
annualburnnon_modis_df = pd.DataFrame(annualburnnon, index=years2, columns=ecoprov.province.values)

In [ ]:
# export MODIS
annualburnall_modis_df.to_csv('west_US_fires_modis_ecoprovinces.txt')
annualburnfor_modis_df.to_csv('west_US_fires_modis_ecoprovinces_forest.txt')
annualburnnon_modis_df.to_csv('west_US_fires_modis_ecoprovinces_nonforest.txt')

In [ ]:
# clear some variables to free up storage
del modis
del modisforest
del modisnon
del annualburnall
del annualburnfor
del annualburnnon
del annualburnall_modis_df
del annualburnfor_modis_df
del annualburnnon_modis_df

In [5]:
# import burned area (WUMI1 + WUMI2)
#burnarea_data_string = directory + '1km\\wildfire\\burnarea_combined_WUMI1WUMI2.nc'
burnarea_data_string = directory + '1km//wildfire//burnarea_combined_WUMI1WUMI2.nc'
burnarea_data = xr.open_dataset(burnarea_data_string, engine='netcdf4')
burnarea = burnarea_data.burnarea
burnarea_data = [] # clear import

# import forest area
#forest_string = directory + '1km\\landcover\\US_ForestType_Ruefenacht\\forest_type_frac.nc'
forest_string = directory + '1km//landcover//US_ForestType_Ruefenacht//forest_type_frac.nc'
forest = xr.open_dataset(forest_string, engine='netcdf4')
forest = forest.forest_type_frac.sum(dim='ftype') # sum over all forest types

# calculate burned area in forests
burnforest = burnarea*forest

# calculate non-forest
burnnon = burnarea - burnforest

# clear forest
del forest

In [14]:
# find sum of burned area for each ecoprovince (WUMI)

firstyear = pd.DatetimeIndex(burnarea.time.values).year.min()
finalyear = pd.DatetimeIndex(burnarea.time.values).year.max()
years = np.arange(firstyear, finalyear+1, 1)

# initialize list
annualburnall = np.zeros([len(years), len(ecoprov.province)])
annualburnfor = np.zeros([len(years), len(ecoprov.province)])
annualburnnon = np.zeros([len(years), len(ecoprov.province)])

# this takes a while because it's at 1km resolution
counter = 0
for thisregion in ecoprov:
    annualburnall[:,counter] = annual_seasonArea(burnarea, thisregion)
    annualburnfor[:,counter] = annual_seasonArea(burnforest, thisregion)
    annualburnnon[:,counter] = annual_seasonArea(burnnon, thisregion)
    
    print('province '+str(counter)+'/'+str((len(ecoprov.province)-1))) # progress
    counter+=1

# create pandas dataframe
annualburnall_df = pd.DataFrame(annualburnall, index=years, columns=ecoprov.province.values)
annualburnfor_df = pd.DataFrame(annualburnfor, index=years, columns=ecoprov.province.values)
annualburnnon_df = pd.DataFrame(annualburnnon, index=years, columns=ecoprov.province.values)

province 0/20
province 1/20
province 2/20
province 3/20
province 4/20
province 5/20
province 6/20
province 7/20
province 8/20
province 9/20
province 10/20
province 11/20
province 12/20
province 13/20
province 14/20
province 15/20
province 16/20
province 17/20
province 18/20
province 19/20
province 20/20


In [15]:
# export WUMI
annualburnall_df.to_csv('west_US_fires_ecoprovinces.txt')
annualburnfor_df.to_csv('west_US_fires_ecoprovinces_forest.txt')
annualburnnon_df.to_csv('west_US_fires_ecoprovinces_nonforest.txt')

In [16]:
# clear some variables to free up storage
del burnarea
del burnforest
del burnnon
del annualburnall
del annualburnfor
del annualburnnon
del annualburnall_df
del annualburnfor_df
del annualburnnon_df

In [17]:
# last export date and time
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Data last updated =", dt_string)

Data last updated = 07/07/2023 20:30:43
